# Common Libraries

In [2]:
import os, sys, shutil
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.gridspec import GridSpec
from ipywidgets import interact, IntSlider, FloatSlider, Layout, Button, Output, VBox, HBox

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

# Custom Libraries

In [5]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from mujoco_util import get_joints, get_simulated_img, get_dependent_joints

# Params

In [15]:
model_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_emg/Myosuite/myoarm_mocap/myoarm_mocap.xml"

simulation_interval = 1 / 30

# Initialize model

In [16]:
model = mujoco.MjModel.from_xml_path(model_path)
model.opt.timestep = simulation_interval
model.opt.gravity = [0,0,0]
mj_data = mujoco.MjData(model)

# Load data

In [18]:
# Joint
joint_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, idx) for idx in range(model.njnt)]
joint_info_df = get_joints(model_path)
dependent_joint_names = joint_info_df.loc[:, joint_info_df.loc["dependent"] == True].columns.to_numpy()
independent_joint_names = joint_info_df.loc[:, joint_info_df.loc["independent"] == True].columns.to_numpy()

target_joint_names = independent_joint_names

# Renderer

In [20]:
# Renderer
renderer = mujoco.Renderer(model, height=400, width=600)
scene_option = mujoco.MjvOption()
scene_option.frame = mujoco.mjtFrame.mjFRAME_WORLD

# Camera
camera = mujoco.MjvCamera()
camera.azimuth = -90
camera.elevation = -90
camera.distance = 2.5
camera.lookat = np.array([0, 0, 0])

# Torque simulation

In [24]:
show_interval = 1
n_step = 5

key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, "default-pose")
mujoco.mj_resetDataKeyframe(model, mj_data, key_id)
mujoco.mj_forward(model, mj_data)
    
dof_addrs = {}
for j_name in target_joint_names:
    jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, j_name)
    dof_addrs[j_name] = model.jnt_dofadr[jid]

sliders = {}
for j_name in target_joint_names:
    sliders[j_name] = FloatSlider(
        value=0.0,
        min=-5.0,  # 필요에 따라 범위를 조절하세요
        max=5.0,
        step=1.0,
        description=f'{j_name}:',
        style={'description_width': 'initial'}, # 이름이 잘리지 않도록 설정
        layout=Layout(width="300px")
    )

apply_btn = Button(description = 'Apply Forces & Step', button_style = 'success')
reset_btn = Button(description = 'Reset', button_style = 'danger')
output_area = Output()

qpos_history = []
applied_torques = []
def show_frame():
    with output_area:
        output_area.clear_output(wait=True)
        renderer.update_scene(mj_data, camera = camera, scene_option = scene_option)

        fig, axis = plt.subplots(1, figsize = (8, 6))
        axis.imshow(renderer.render())
        axis.axis("off")
        plt.tight_layout()
        plt.show()

def on_apply_btn_clicked(b):
    for step_i in range(n_step):
        for j_name in target_joint_names:
            addr = dof_addrs[j_name]
            mj_data.qfrc_applied[addr] = sliders[j_name].value
    
        applied_torques.append(mj_data.qfrc_applied.copy())
        qpos_history.append(mj_data.qpos.copy())
        mujoco.mj_step(model, mj_data)

        if (step_i % show_interval) == 0:
            show_frame()

def on_reset_btn_clicked(b):
    qpos_history = []
    applied_torques = []
    for slider in sliders.values():
        slider.value = 0.0
    
    key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, "default-pose")
    mujoco.mj_resetDataKeyframe(model, mj_data, key_id)
    mujoco.mj_forward(model, mj_data)
        
    mujoco.mj_forward(model, mj_data)
    show_frame()

apply_btn.on_click(on_apply_btn_clicked)
reset_btn.on_click(on_reset_btn_clicked)

slider_list = list(sliders.values())
split_sliders = np.array_split(slider_list, 3)
sliders_box = HBox([
    VBox(list(split_sliders[0])), 
    VBox(list(split_sliders[1])), 
    VBox(list(split_sliders[2]))
])
buttons_box = HBox([apply_btn, reset_btn])
ui_panel = VBox([sliders_box, buttons_box])

display(ui_panel, output_area)

show_frame()

Output()